In [ ]:
import pandas as pd
import numpy as np
import os

print("Starting up the Air Quality data compiler...")

file_map = {
    "India": "/content/IndiaPM25-V6GL0203-Annual-REGIONAL-1998-2023-wThresFrac.csv",
    "Canada": "/content/CanadaPM25-V6GL0203-Annual-REGIONAL-1998-2023-wThresFrac.csv",
    "China": "/content/ChinaPM25-V6GL0203-Annual-REGIONAL-1998-2023-wThresFrac.csv",
    "USA": "/content/UnitedStatesPM25-V6GL0203-Annual-REGIONAL-1998-2023-wThresFrac.csv",
    "Global": "/content/GlobalPM25-V6GL0203-Annual-1998-2023-wThresFrac.csv"
}

all_dfs = []
print("Loading datasets from around the world...")

for country, filename in file_map.items():
    if os.path.exists(filename):
        temp_df = pd.read_csv(filename)
        temp_df['Source'] = country
        all_dfs.append(temp_df)
        print(f"  ✓ Successfully loaded {country} data.")
    else:
        print(f"Couldn't find the file for {country} at {filename}.")

if not all_dfs:
    print("\nError: No data found. Please make sure your CSV files are uploaded to Colab!")
else:
    master_df = pd.concat(all_dfs, ignore_index=True)

    target_col = 'Population-Weighted PM2.5 [ug/m3]'

    master_df = master_df.sort_values(by=['Source', 'Region', 'Year']).reset_index(drop=True)

    master_df['pm_lag_1'] = master_df.groupby(['Source', 'Region'])[target_col].shift(1)

    master_df['pm_lag_2'] = master_df.groupby(['Source', 'Region'])[target_col].shift(2)

    master_df['pm_lag_3'] = master_df.groupby(['Source', 'Region'])[target_col].shift(3)

    master_df['pm_roll_3'] = master_df[['pm_lag_1', 'pm_lag_2', 'pm_lag_3']].mean(axis=1)

    columns_to_keep = [
        'Source',
        'Year',
        'Region',
        'Geographic-Mean PM2.5 [ug/m3]',
        'Population Coverage [%]',
        'Geographic Coverage [%]',
        'pm_lag_1',
        'pm_lag_2',
        'pm_lag_3',
        'pm_roll_3',
        target_col
    ]

    ts_df = master_df[columns_to_keep].copy()

    ts_df = ts_df.dropna().reset_index(drop=True)

    ts_df = ts_df.rename(columns={target_col: 'Target_PM25_Current_Year'})

    print("\n Data transformation complete! Here is a sneak peek at your new Time Series dataset:")
    display(ts_df.head(10))

Starting up the Air Quality data compiler...
Loading datasets from around the world...
  ✓ Successfully loaded India data.
  ✓ Successfully loaded Canada data.
  ✓ Successfully loaded China data.
  ✓ Successfully loaded USA data.
  ✓ Successfully loaded Global data.

 Data transformation complete! Here is a sneak peek at your new Time Series dataset:


,Source,Year,Region,Geographic-Mean PM2.5 [ug/m3],Population Coverage [%],Geographic Coverage [%],pm_lag_1,pm_lag_2,pm_lag_3,pm_roll_3,Target_PM25_Current_Year
0,Canada,2001,Alberta,4.5,100.0,100.0,7.2,6.9,9.3,7.800000,6.7
1,Canada,2002,Alberta,5.6,100.0,100.0,6.7,7.2,6.9,6.933333,7.0
2,Canada,2003,Alberta,5.5,100.0,100.0,7.0,6.7,7.2,6.966667,8.0
3,Canada,2004,Alberta,5.5,100.0,100.0,8.0,7.0,6.7,7.233333,6.9
4,Canada,2005,Alberta,4.7,100.0,100.0,6.9,8.0,7.0,7.300000,6.7
5,Canada,2006,Alberta,5.3,100.0,100.0,6.7,6.9,8.0,7.200000,7.2
6,Canada,2007,Alberta,5.1,100.0,100.0,7.2,6.7,6.9,6.933333,7.0
7,Canada,2008,Alberta,5.3,100.0,100.0,7.0,7.2,6.7,6.966667,7.1
8,Canada,2009,Alberta,5.3,100.0,100.0,7.1,7.0,7.2,7.100000,7.6
9,Canada,2010,Alberta,6.0,100.0,100.0,7.6,7.1,7.0,7.233333,8.4


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

print(" Step 1: Prepping the data for training...")

features = [
    'Year',
    'Geographic-Mean PM2.5 [ug/m3]',
    'Population Coverage [%]',
    'Geographic Coverage [%]',
    'pm_lag_1',
    'pm_lag_2',
    'pm_lag_3',
    'pm_roll_3'
]
target = 'Target_PM25_Current_Year'

X = ts_df[features]
y = ts_df[target]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n Step 2: Training the Random Forest model. Stand by...")

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_val_pred = model.predict(X_val)

r2 = r2_score(y_val, y_val_pred)
mae = mean_absolute_error(y_val, y_val_pred)

print(f"   Accuracy (R² Score): {r2 * 100:.2f}%")
print(f"   Margin of Error (MAE): It's usually off by about {mae:.2f} ug/m3")

test_file_path = '/content/Test_Features.csv'

print("\n Step 3: Processing the official Test Dataset...")
if os.path.exists(test_file_path):
    test_df = pd.read_csv(test_file_path)

    test_df = test_df.rename(columns={
        'pm_lag1': 'pm_lag_1',
        'pm_lag2': 'pm_lag_2',
        'pm_lag3': 'pm_lag_3',
        'pm_roll3': 'pm_roll_3'
    })

    missing_cols = [col for col in features if col not in test_df.columns]

    if missing_cols:
        print(f" The test dataset is missing some expected columns: {missing_cols}")
    else:
        X_test_final = test_df[features]

        final_predictions = model.predict(X_test_final)

        test_df['Predicted_PM25'] = final_predictions

        print("  ✓ Predictions successfully generated!")
        print("\nHere's a sneak peek at the first 5 predictions:")
        display(test_df[['Year', 'Region', 'Predicted_PM25']].head())

        output_filename = '/content/Final_Test_Predictions.csv'
        test_df.to_csv(output_filename, index=False)
        print(f"\n  Saved your predictions to: {output_filename}")

else:
    print(f" Couldn't find {test_file_path}. Did you upload it to your Colab environment?")

 Step 1: Prepping the data for training...

 Step 2: Training the Random Forest model. Stand by...
   Accuracy (R² Score): 98.29%
   Margin of Error (MAE): It's usually off by about 1.21 ug/m3

 Step 3: Processing the official Test Dataset...
  ✓ Predictions successfully generated!

Here's a sneak peek at the first 5 predictions:


,Year,Region,Predicted_PM25
0,2021,Afghanistan,34.954
1,2022,Afghanistan,34.984
2,2023,Afghanistan,38.511
3,2024,Afghanistan,37.432
4,2021,Akrotiri and Dhekelia,13.627



  Saved your predictions to: /content/Final_Test_Predictions.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import os

print("Starting up the Air Quality predictor...")

print("Training the model on historical data. This might take a second...")

features = [
    'Year',
    'Geographic-Mean PM2.5 [ug/m3]',
    'Population Coverage [%]',
    'Geographic Coverage [%]',
    'pm_lag_1',
    'pm_lag_2',
    'pm_lag_3',
    'pm_roll_3'
]
target = 'Target_PM25_Current_Year'

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(ts_df[features], ts_df[target])


def categorize_pm25(val):
    if val <= 12.0:
        return 'Good'
    elif val <= 35.4:
        return 'Moderate'
    elif val <= 55.4:
        return 'Sensitive'
    else:
        return 'Critical'


test_file_path = '/content/Test_Features.csv'

if os.path.exists(test_file_path):
    print(f" Found the test file at {test_file_path}. Processing now...")

    test_df = pd.read_csv(test_file_path)

    test_df = test_df.rename(columns={
        'pm_lag1': 'pm_lag_1',
        'pm_lag2': 'pm_lag_2',
        'pm_lag3': 'pm_lag_3',
        'pm_roll3': 'pm_roll_3'
    })

    X_test_final = test_df[features]

    raw_predictions = model.predict(X_test_final)

    class_predictions = [categorize_pm25(val) for val in raw_predictions]

    final_output = pd.DataFrame({
        'Year': test_df['Year'],
        'Region': test_df['Region'],
        'Predicted_Air_Quality': class_predictions
    })

    print("\n Here's a look at the predictions:")

    pd.set_option('display.max_rows', None)
    display(final_output)
    pd.reset_option('display.max_rows')

    save_path = '/content/Final_Classified_Predictions.csv'
    final_output.to_csv(save_path, index=False)
    print(f"\n Success! Your classified predictions are saved to: {save_path}")

else:

    print(f"I couldn't find {test_file_path}. Did you forget to upload it to Colab?")

Starting up the Air Quality predictor...
Training the model on historical data. This might take a second...
 Found the test file at /content/Test_Features.csv. Processing now...

 Here's a look at the predictions:


,Year,Region,Predicted_Air_Quality
0,2021,Afghanistan,Moderate
1,2022,Afghanistan,Moderate
2,2023,Afghanistan,Sensitive
3,2024,Afghanistan,Sensitive
4,2021,Akrotiri and Dhekelia,Moderate
5,2022,Akrotiri and Dhekelia,Moderate
6,2023,Akrotiri and Dhekelia,Moderate
7,2024,Akrotiri and Dhekelia,Moderate
8,2021,Aland,Good
9,2022,Aland,Good



 Success! Your classified predictions are saved to: /content/Final_Classified_Predictions.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
import os

print("Step 1: Quickly training the model...")
features = ['Year', 'Geographic-Mean PM2.5 [ug/m3]', 'Population Coverage [%]',
            'Geographic Coverage [%]', 'pm_lag_1', 'pm_lag_2', 'pm_lag_3', 'pm_roll_3']

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(ts_df[features], ts_df['Target_PM25_Current_Year'])

print(" Step 2: Loading test data and making predictions...")
test_df = pd.read_csv('/content/Test_Features.csv')

test_df = test_df.rename(columns={'pm_lag1': 'pm_lag_1', 'pm_lag2': 'pm_lag_2',
                                  'pm_lag3': 'pm_lag_3', 'pm_roll3': 'pm_roll_3'})

test_df = test_df.sort_values(by=['Region', 'Year']).reset_index(drop=True)

test_df['Predicted_PM25_Number'] = model.predict(test_df[features])

print(" Step 3: Reverse-engineering the true answers from the future...")

test_df['True_PM25_Number'] = test_df.groupby('Region')['pm_lag_1'].shift(-1)

eval_df = test_df.dropna(subset=['True_PM25_Number']).copy()

print(" Step 4: Translating numbers to EPA categories and grading the test...")

def categorize_pm25(val):
    if val <= 12.0: return 'Good'
    elif val <= 35.4: return 'Moderate'
    elif val <= 55.4: return 'Sensitive'
    else: return 'Critical'

eval_df['Predicted_Class'] = eval_df['Predicted_PM25_Number'].apply(categorize_pm25)
eval_df['True_Class'] = eval_df['True_PM25_Number'].apply(categorize_pm25)

class_accuracy = accuracy_score(eval_df['True_Class'], eval_df['Predicted_Class'])
r2 = r2_score(eval_df['True_PM25_Number'], eval_df['Predicted_PM25_Number'])
mae = mean_absolute_error(eval_df['True_PM25_Number'], eval_df['Predicted_PM25_Number'])

print("\n" + "=" * 45)
print(" FINAL REPORT CARD (TEST DATA EVALUATION)")
print("=" * 45)
print(f"Category Accuracy:  {class_accuracy * 100:.2f}% (How often did we guess the right text label?)")
print(f"Numeric Accuracy:   {r2 * 100:.2f}% (How close were our raw numbers?)")
print(f"Average Error:      Off by {mae:.2f} ug/m3 on average")
print("=" * 45)


Step 1: Quickly training the model...
 Step 2: Loading test data and making predictions...
 Step 3: Reverse-engineering the true answers from the future...
 Step 4: Translating numbers to EPA categories and grading the test...

 FINAL REPORT CARD (TEST DATA EVALUATION)
Category Accuracy:  96.75% (How often did we guess the right text label?)
Numeric Accuracy:   98.69% (How close were our raw numbers?)
Average Error:      Off by 0.95 ug/m3 on average


In [ ]:
import joblib
from google.colab import files

print("Time to pack everything up and ship it to your computer...")

model_filename = 'pm25_random_forest_model.pkl'
joblib.dump(model, model_filename)
print(f"  ✓ Success! Your trained model is safely packaged as '{model_filename}'")

report_filename = 'Model_Accuracy_Report.txt'

with open(report_filename, 'w') as f:
    f.write(" PM2.5 MODEL EVALUATION REPORT 📊\n")
    f.write("=" * 45 + "\n")
    f.write(f"Category Accuracy:  {class_accuracy * 100:.2f}%\n")
    f.write(f"Numeric Accuracy:   {r2 * 100:.2f}%\n")
    f.write(f"Average Error:      {mae:.2f} ug/m3\n")
    f.write("=" * 45 + "\n")
    f.write("Note: Accuracy was calculated using the 'future-lag' reverse-engineering trick.\n")

print(f"  ✓ Success! Report card saved as '{report_filename}'")

print("\n Initiating downloads... (Keep an eye on your browser's download queue!)")
print("Note: If your browser asks for permission to download multiple files, make sure to click 'Allow'.")


files.download(model_filename)

files.download(report_filename)

files.download('/content/Final_Classified_Predictions_with_Accuracy.csv')

Time to pack everything up and ship it to your computer...
  ✓ Success! Your trained model is safely packaged as 'pm25_random_forest_model.pkl'
  ✓ Success! Report card saved as 'Model_Accuracy_Report.txt'

 Initiating downloads... (Keep an eye on your browser's download queue!)
Note: If your browser asks for permission to download multiple files, make sure to click 'Allow'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: /content/Final_Classified_Predictions_with_Accuracy.csv